# Análisis temporal de nueve finalistas
Carga `backtesting_results.json` de la corrida consolidada. El gráfico incluye una referencia horizontal en el AUC de entrenamiento inicial de cada modelo. Los cuatro meses participan en selección; no constituyen una prueba independiente del ganador final.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
RESULTS_PATH = '' # Opcional: ruta local. En Colab se solicita el archivo.
if RESULTS_PATH:
    raw=Path(RESULTS_PATH).read_bytes()
else:
    from google.colab import files
    uploaded=files.upload()
    raw=next(iter(uploaded.values()))
try: results=json.loads(raw.decode('utf-8-sig'))
except UnicodeDecodeError: results=json.loads(raw.decode('cp1252'))
print('Campeón inicial:',results['initial_champion'])
print('Campeón temporal:',results['champion'])
print(results['selection_reason'])


In [ ]:
ranking=pd.DataFrame([{'modelo':name,'auc_train':b['reference_auc_train'],
    'auc_promedio':b['avg_auc_backtest'],'decay_promedio_pct':b['avg_decay_pct'],
    'olas_validas':b['valid_waves'],'alerta':b['drift_alert'],'drift_severo':b['severe_drift']}
    for name,b in results['models'].items()])
display(ranking.sort_values('decay_promedio_pct'))
waves=pd.DataFrame([{'modelo':name,**w} for name,b in results['models'].items() for w in b['waves']])
display(waves[['modelo','partition','auc','decay_pct','auc_drop_pp','n_obs','drift_alert','prediction_seconds']])

In [ ]:
fig,axes=plt.subplots(3,3,figsize=(15,11),sharey=True)
for ax,(name,b) in zip(axes.flat,results['models'].items()):
    rows=b['waves']; x=[w['wave'] for w in rows]
    ax.plot(x,[w['auc'] if w['auc'] is not None else np.nan for w in rows],marker='o',label='AUC mensual')
    ax.axhline(b['reference_auc_train'],color='tab:orange',ls='--',label='AUC entrenamiento')
    ax.axhline(.5,color='gray',ls=':')
    ax.set_xticks(x,[w['partition'] for w in rows]);ax.set_ylim(0,1.03)
    ax.set_title(name);ax.set_ylabel('AUC ROC');ax.grid(alpha=.25);ax.legend(fontsize=8)
fig.tight_layout();fig.savefig('auc_nueve_modelos.png',dpi=160,bbox_inches='tight');plt.show()

Las alertas comparan la caída absoluta en puntos de AUC con el umbral configurado. El decay es relativo al AUC de entrenamiento. Una caída del AUC no identifica por sí sola concept drift. La selección considera filtros P1, cuatro olas definidas y menor decay promedio; los candidatos con alertas en las cuatro olas se excluyen.